In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.pipeline import Pipeline
import category_encoders as ce  # для TargetEncoder

import warnings
warnings.filterwarnings('ignore')

In [45]:
np.random.seed(42)
torch.manual_seed(42)

In [46]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"
columns = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df = pd.read_csv(url, names=columns)

In [47]:
print(df.info())
print('-'*40)
print(' ')
print(df.head())
print(' ')
print('-'*40)
print(' ')
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728 entries, 0 to 1727
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   buying    1728 non-null   object
 1   maint     1728 non-null   object
 2   doors     1728 non-null   object
 3   persons   1728 non-null   object
 4   lug_boot  1728 non-null   object
 5   safety    1728 non-null   object
 6   class     1728 non-null   object
dtypes: object(7)
memory usage: 94.6+ KB
None
----------------------------------------
 
  buying  maint doors persons lug_boot safety  class
0  vhigh  vhigh     2       2    small    low  unacc
1  vhigh  vhigh     2       2    small    med  unacc
2  vhigh  vhigh     2       2    small   high  unacc
3  vhigh  vhigh     2       2      med    low  unacc
4  vhigh  vhigh     2       2      med    med  unacc
 
----------------------------------------
 
       buying  maint doors persons lug_boot safety  class
count    1728   1728  1728    1728     172

In [48]:
df['class'].value_counts()

class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64

In [49]:
X = df.drop('class', axis=1)
y = df['class']

In [50]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [51]:
class_names = le.classes_
num_classes = len(class_names)

In [52]:
X_train,X_test,y_train,y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42,stratify=y_encoded
)

In [53]:
buying_order = ['vhigh','high','med','low']
maint_order = ['vhigh', 'high', 'med', 'low']
doors_order = ['2', '3', '4', '5more']  # 5more > 4
persons_order = ['2', '4', 'more']
lug_boot_order = ['small', 'med', 'big']
safety_order = ['low', 'med', 'high']
ordinal_category = [buying_order,maint_order,doors_order,persons_order,lug_boot_order,safety_order]


In [61]:
data_version = {}
ordinal_enc = OrdinalEncoder(categories=ordinal_category)
X_train_ord = ordinal_enc.fit_transform(X_train)
X_test_ord = ordinal_enc.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ord)
X_test_scaled = scaler.transform(X_test_ord)
data_version['StandardScaler'] = (X_train_scaled, X_test_scaled)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_ord)
X_test_scaled = scaler.transform(X_test_ord)
data_version['MinMaxScaler'] = (X_train_scaled, X_test_scaled)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_ord)
X_test_scaled = scaler.transform(X_test_ord)
data_version['RobustScaler'] = (X_train_scaled, X_test_scaled)

ordinal_ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_ohe = ordinal_ohe.fit_transform(X_train)
X_test_ohe = ordinal_ohe.transform(X_test)
data_version['ohe'] =(X_train_ohe, X_test_ohe)

scaler = StandardScaler()
X_train_ohe_scaled = scaler.fit_transform(X_train_ohe)
X_test_ohe_scaled = scaler.transform(X_test_ohe)
data_version['StandardScaler_ohe'] = (X_train_ohe_scaled, X_test_ohe_scaled)

for k in data_version:
    print(f'{k} : {data_version[k][0].shape}')


StandardScaler : (1382, 6)
MinMaxScaler : (1382, 6)
RobustScaler : (1382, 6)
ohe : (1382, 15)
StandardScaler_ohe : (1382, 15)
